In [52]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [53]:
words = open('names.txt').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [54]:
len(words)

32033

In [55]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = { s:i+1 for i,s in enumerate(chars) } # stoi: string
stoi['.'] = 0 # padding character
itos = { i:s for s,i in stoi.items() } # itos: integer to string
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [348]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
    # print(w)
    context = [0] * block_size
    for ch in w + '.': # we want to predict the special end character
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(''.join(itos[i] for i in context), '->', itos[ix])
        context = context[1:] + [ix] # crop and append
X = torch.tensor(X)
Y = torch.tensor(Y)

In [349]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [350]:
C = torch.randn((27, 2))

In [351]:
C[5]
# 索引更快，所以放弃独热编码
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.3251, 0.1783])

In [352]:
# 直接索引矩阵，完成词嵌入
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [353]:
W1 = torch.randn((6, 100)) # 输入特征维是 3 个词 * 2 维嵌入
b1 = torch.randn(100)

In [354]:
# 三种维度重排方法，但 view 效率最高
# torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], dim=1)
# torch.cat(torch.unbind(emb, dim=1), dim=1)
emb.view(-1, 6).shape

torch.Size([228146, 6])

In [355]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [356]:
h.shape

torch.Size([228146, 100])

In [357]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [358]:
logits = h @ W2 + b2
logits.shape

torch.Size([228146, 27])

In [359]:
counts = logits.exp()

In [360]:
prob = counts / counts.sum(1, keepdim=True)

In [364]:
loss = -(prob[torch.arange(prob.shape[0]), Y]).log().mean()
loss

tensor(17.6039)

In [365]:
# 出错点：手动计算交叉熵会导致数值上/下溢出
logits = torch.tensor([-1, 2, 10000])
counts = logits.exp()
prob = counts / counts.sum(0)
loss = -(prob[torch.tensor([1])]).log()
loss

tensor([inf])

重构

In [435]:
# build the dataset
def build_dataset(words):
    block_size = 3 # context length: how many characters do we take to predict the next one?
    X, Y = [], []
    for w in words:
        # print(w)
        context = [0] * block_size
        for ch in w + '.': # we want to predict the special end character
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '->', itos[ix])
            context = context[1:] + [ix] # crop and append
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [542]:
# Params
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 32), generator=g)
W1 = torch.randn((32*3, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True
sum(p.numel() for p in parameters)

25691

In [543]:
# lre = torch.linspace(-3, 0, steps=30000)
# lrs = 10**lre
# lri = []
# lossi = []
# stepi = []

In [547]:
optimizer = torch.optim.AdamW(parameters, lr=0.001, weight_decay=0.1)

In [548]:
for i in range(50000):

    # minibatch (accelerate training)
    ix = torch.randint(0, Xtr.shape[0], (128,))

    # forward pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 32*3) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])


    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # update
    # lr = lrs[i]
    optimizer.step()
    

    # track stats
    # stepi.append(i)
    # lri.append(lr)
    # lossi.append(loss.log10().item())

  
print(loss.item())

2.2502799034118652


In [550]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 32*3) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
print(loss.item())

2.1554322242736816


In [421]:
# plt.plot(lre, lossi)

In [480]:
# plt.figure(figsize=(8,8))
# plt.scatter(C[:,0].data, C[:,1].data, s=200)
# for i in range(C.shape[0]):
#     plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha='center', va='center', color='white')
# plt.grid('minor')